# 03.4 — Edit Distance & Spell Correction

**Problem it solves:** How similar are two strings? How do you correct a misspelling?

**Levenshtein distance:** Minimum number of single-character edits (insert, delete, substitute) to transform one string into another. Classic dynamic programming.

---

In [ ]:
import numpy as np

def levenshtein(s1: str, s2: str, 
                ins_cost=1, del_cost=1, sub_cost=1) -> int:
    """
    Levenshtein (edit) distance using dynamic programming.
    
    DP table: dp[i][j] = edit distance between s1[:i] and s2[:j]
    Base cases: dp[i][0] = i (delete all), dp[0][j] = j (insert all)
    Recurrence:
      if s1[i-1] == s2[j-1]: dp[i][j] = dp[i-1][j-1]   (no cost)
      else: dp[i][j] = 1 + min(dp[i-1][j],   # delete from s1
                                dp[i][j-1],   # insert into s1
                                dp[i-1][j-1]) # substitute
    """
    m, n = len(s1), len(s2)
    dp = np.zeros((m+1, n+1), dtype=int)
    
    # Base cases
    for i in range(m+1): dp[i][0] = i * del_cost
    for j in range(n+1): dp[0][j] = j * ins_cost
    
    for i in range(1, m+1):
        for j in range(1, n+1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]  # match — no cost
            else:
                dp[i][j] = min(
                    dp[i-1][j] + del_cost,    # delete s1[i-1]
                    dp[i][j-1] + ins_cost,    # insert s2[j-1]
                    dp[i-1][j-1] + sub_cost,  # substitute
                )
    
    return int(dp[m][n])

def levenshtein_backtrace(s1: str, s2: str) -> tuple:
    """Return edit distance AND the alignment (sequence of ops)."""
    m, n = len(s1), len(s2)
    dp = np.zeros((m+1, n+1), dtype=int)
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1,m+1):
        for j in range(1,n+1):
            if s1[i-1]==s2[j-1]: dp[i][j]=dp[i-1][j-1]
            else: dp[i][j]=1+min(dp[i-1][j],dp[i][j-1],dp[i-1][j-1])
    
    # Backtrace
    ops = []
    i, j = m, n
    while i > 0 or j > 0:
        if i > 0 and j > 0 and s1[i-1] == s2[j-1]:
            ops.append(f'MATCH({s1[i-1]})')
            i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            ops.append(f'SUB({s1[i-1]}->{s2[j-1]})')
            i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            ops.append(f'DEL({s1[i-1]})')
            i -= 1
        else:
            ops.append(f'INS({s2[j-1]})')
            j -= 1
    
    return int(dp[m][n]), list(reversed(ops))

# Examples
pairs = [
    ('kitten', 'sitting'),
    ('saturday', 'sunday'),
    ('recieve', 'receive'),  # common misspelling
    ('teh', 'the'),
    ('', 'hello'),
    ('hello', 'hello'),
]

for s1, s2 in pairs:
    dist = levenshtein(s1, s2)
    _, ops = levenshtein_backtrace(s1, s2)
    print(f"  {s1!r:15} -> {s2!r:15}  dist={dist}  ops={ops}")

In [ ]:
# Spell corrector using edit distance
# Based on Peter Norvig's famous implementation

from collections import Counter

class SpellCorrector:
    """
    Noisy channel model for spell correction:
    argmax_c P(c|w) = argmax_c P(w|c) * P(c)
    
    P(c) = unigram probability (language model)
    P(w|c) = error model (approximated by edit distance)
    """
    
    def __init__(self, max_edit_distance: int = 2):
        self.max_edit = max_edit_distance
        self.word_counts: Counter = Counter()
    
    def train(self, text: str):
        import re
        words = re.findall(r'[a-z]+', text.lower())
        self.word_counts = Counter(words)
        return self
    
    def _candidates_edit1(self, word: str) -> set:
        """All strings edit-distance 1 from word."""
        letters = 'abcdefghijklmnopqrstuvwxyz'
        splits     = [(word[:i], word[i:]) for i in range(len(word) + 1)]
        deletes    = {L + R[1:]        for L, R in splits if R}
        transposes = {L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1}
        replaces   = {L + c + R[1:]   for L, R in splits if R for c in letters}
        inserts    = {L + c + R       for L, R in splits for c in letters}
        return deletes | transposes | replaces | inserts
    
    def _candidates_edit2(self, word: str) -> set:
        return {e2 for e1 in self._candidates_edit1(word)
                   for e2 in self._candidates_edit1(e1)}
    
    def _known(self, words: set) -> set:
        return {w for w in words if w in self.word_counts}
    
    def correct(self, word: str) -> str:
        word = word.lower()
        # Priority: exact match > edit-1 > edit-2 > original
        candidates = (
            self._known({word}) or
            self._known(self._candidates_edit1(word)) or
            (self._known(self._candidates_edit2(word)) if self.max_edit >= 2 else set()) or
            {word}
        )
        # Score by unigram probability
        total = sum(self.word_counts.values())
        return max(candidates, key=lambda w: self.word_counts.get(w, 0) / total)
    
    def correct_with_suggestions(self, word: str, n=5) -> list:
        word = word.lower()
        candidates = (
            self._known({word}) or
            self._known(self._candidates_edit1(word)) or
            self._known(self._candidates_edit2(word)) or {word}
        )
        total = sum(self.word_counts.values())
        return sorted(candidates, key=lambda w: self.word_counts.get(w,0)/total, reverse=True)[:n]

# Train on a corpus
corpus = """
the quick brown fox jumps over the lazy dog the dog barked
machine learning natural language processing text classification
computer science programming language algorithm function variable
receive believe achieve perceive conceive december february
government parliament democracy election political candidate
beautiful wonderful terrible horrible excellent magnificent
the the the a a a in in in is is is of of of and and and
"""
corrector = SpellCorrector(max_edit_distance=2).train(corpus)

misspellings = ['recieve', 'teh', 'langauge', 'machien', 'leraning', 
                'goverment', 'beautifull', 'algoritm', 'programing']

print("Spell corrections:")
for word in misspellings:
    correction = corrector.correct(word)
    suggestions = corrector.correct_with_suggestions(word, n=3)
    print(f"  {word:20} -> {correction:20} (top suggestions: {suggestions})")

In [ ]:
# Damerau-Levenshtein: adds transposition as a single operation
# (ab -> ba costs 1, not 2) — transpositions are the most common typo

def damerau_levenshtein(s1: str, s2: str) -> int:
    m, n = len(s1), len(s2)
    dp = np.zeros((m+2, n+2), dtype=int)
    max_dist = m + n
    dp[0][0] = max_dist
    for i in range(m+1): dp[i+1][0] = max_dist; dp[i+1][1] = i
    for j in range(n+1): dp[0][j+1] = max_dist; dp[1][j+1] = j
    
    da = {}  # last position of each character in s1
    for i in range(1, m+1):
        db = 0
        for j in range(1, n+1):
            i1 = da.get(s2[j-1], 0)
            j1 = db
            cost = 0 if s1[i-1] == s2[j-1] else 1
            if cost == 0: db = j
            dp[i+1][j+1] = min(
                dp[i][j] + cost,       # sub or match
                dp[i+1][j] + 1,        # insert
                dp[i][j+1] + 1,        # delete
                dp[i1][j1] + (i-i1-1) + 1 + (j-j1-1)  # transpose
            )
        da[s1[i-1]] = i
    return int(dp[m+1][n+1])

print("Levenshtein vs Damerau-Levenshtein:")
test_pairs = [('ab', 'ba'), ('abc', 'bca'), ('teh', 'the'), ('recieve', 'receive')]
for s1, s2 in test_pairs:
    lev = levenshtein(s1, s2)
    dam = damerau_levenshtein(s1, s2)
    print(f"  {s1!r:10} vs {s2!r:10}  Lev={lev}  Damerau={dam}")

## Summary

**Edit distance** is used in: spell correction, DNA sequence alignment, fuzzy string matching, keyboard autocorrect, record deduplication.

**Norvig's spell corrector** works remarkably well with just a word frequency list. The key insight: most misspellings are within edit distance 2.

**What breaks it:**
- Real-word errors: 'their/there/they're' — all valid words, need context
- Long words: edit-2 generates millions of candidates
- Languages with agglutination (Finnish, Turkish) have very different error distributions
- Context-insensitive: doesn't know if 'to/too/two' is right for this sentence

---
**Next:** 03.5 — CRF for Sequence Labeling